# 05 - Build, Run, and Schedule the H2O Batch Scoring Pipeline

This self-contained notebook turns the validated H2O binary-model contract into an Azure Machine Learning pipeline. It regenerates all temporary scoring files, proves golden parity locally, runs the pipeline in Dev, writes outputs to ADLS Gen2, inspects job and MLflow telemetry, creates a native schedule, proves one scheduled run, and disables the test schedule.

The notebook does not import project helper modules or depend on Notebook 06. Its only durable prerequisites are the registered model from Notebook 02 and the local model fixture from Notebook 01.

## 1. Configuration and Safety Gates

Copy `.env.example` to `.env` and fill in the shared Azure ML, compute identity, datastore, and schedule values before running this notebook. Authenticate with `az login` or managed identity; do not store credentials in `.env`.

`SUBMIT_TO_AZURE` and `CREATE_TEST_SCHEDULE` independently control Azure mutations. The committed template keeps both disabled. When schedule testing is enabled, `DISABLE_SCHEDULE_AFTER_TEST=true` prevents the test schedule from continuing to create compute jobs.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
import uuid

import mlflow
import numpy as np
import pandas as pd
from azure.ai.ml import Input, MLClient, Output, command
from azure.ai.ml.constants import AssetTypes, TimeZone
from azure.ai.ml.dsl import pipeline
from azure.ai.ml.entities import (
    Data,
    Environment,
    JobSchedule,
    ManagedIdentityConfiguration,
)
from azure.core.exceptions import ResourceNotFoundError
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from mlflow.tracking import MlflowClient

for folder in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    manifest_candidate = folder / "tmp" / "h2o_binary" / "taxi_fare" / "model_manifest.json"
    data_candidate = folder / "data" / "taxi-data" / "raw" / "yellowTaxiData.csv"
    if manifest_candidate.is_file() and data_candidate.is_file():
        REPO_ROOT = folder
        break
else:
    raise FileNotFoundError("Run Notebook 01 and open this notebook inside the MLOPs-AzureML repository")

ENV_FILE = REPO_ROOT / ".env"
load_dotenv(ENV_FILE)

def env_flag(name, default=False):
    value = os.getenv(name)
    if value is None:
        return default

    normalized = value.strip().lower()
    if normalized not in {"1", "0", "true", "false", "yes", "no", "on", "off"}:
        raise ValueError(f"{name} must be true or false")
    return normalized in {"1", "true", "yes", "on"}


CONFIG = {
    "subscription_id": os.getenv("AZURE_SUBSCRIPTION_ID", "").strip(),
    "tenant_id": os.getenv("AZURE_TENANT_ID", "").strip(),
    "resource_group": os.getenv("AZURE_RESOURCE_GROUP", "").strip(),
    "workspace_name": os.getenv("AZUREML_WORKSPACE_NAME", "").strip(),
    "compute_name": os.getenv("AZUREML_COMPUTE_NAME", "").strip(),
    "compute_identity_client_id": os.getenv("AZUREML_COMPUTE_IDENTITY_CLIENT_ID", "").strip(),
    "model_name": os.getenv("AZUREML_MODEL_NAME", "taxi-fare-h2o-binary").strip(),
    "model_version": os.getenv("AZUREML_MODEL_VERSION", "1").strip(),
    "environment_name": "h2o-binary-batch",
    "environment_version": "1",
    "data_name": "taxi-h2o-batch-input",
    "data_version": "1",
    "output_datastore": os.getenv("AZUREML_OUTPUT_DATASTORE", "").strip(),
    "experiment_name": os.getenv(
        "AZUREML_BATCH_EXPERIMENT_NAME", "h2o-binary-batch-scoring"
    ).strip(),
    "schedule_name": os.getenv(
        "AZUREML_BATCH_SCHEDULE_NAME", "h2o-taxi-batch-test-schedule"
    ).strip(),
    "schedule_wait_minutes": 12,
    "h2o_threads": 4,
    "h2o_heap": "6G",
    "submit_to_azure": env_flag("SUBMIT_TO_AZURE"),
    "create_test_schedule": env_flag("CREATE_TEST_SCHEDULE"),
    "disable_schedule_after_test": env_flag("DISABLE_SCHEDULE_AFTER_TEST", True),
    "delete_schedule_after_test": env_flag("DELETE_SCHEDULE_AFTER_TEST"),
}

required_settings = {
    "AZURE_SUBSCRIPTION_ID": CONFIG["subscription_id"],
    "AZURE_RESOURCE_GROUP": CONFIG["resource_group"],
    "AZUREML_WORKSPACE_NAME": CONFIG["workspace_name"],
    "AZUREML_COMPUTE_NAME": CONFIG["compute_name"],
    "AZUREML_COMPUTE_IDENTITY_CLIENT_ID": CONFIG["compute_identity_client_id"],
    "AZUREML_OUTPUT_DATASTORE": CONFIG["output_datastore"],
}
missing_settings = [name for name, value in required_settings.items() if not value]
if missing_settings:
    raise ValueError(
        f"Set {', '.join(missing_settings)} in {ENV_FILE.name} or the process environment"
    )

MODEL_DIR = REPO_ROOT / "tmp" / "h2o_binary" / "taxi_fare"
DATA_PATH = REPO_ROOT / "data" / "taxi-data" / "raw" / "yellowTaxiData.csv"
WORK_DIR = REPO_ROOT / "tmp" / "h2o_batch_pipeline_05"
CODE_DIR = WORK_DIR / "code"
LOCAL_OUTPUT_DIR = WORK_DIR / "local_validation"
DOWNLOAD_DIR = WORK_DIR / "azure_downloads"
SCORE_PATH = CODE_DIR / "score_batch.py"
CONDA_PATH = WORK_DIR / "conda.yaml"
for path in (CODE_DIR, LOCAL_OUTPUT_DIR, DOWNLOAD_DIR):
    path.mkdir(parents=True, exist_ok=True)

credential = AzureCliCredential(tenant_id=CONFIG["tenant_id"] or None)
ml_client = MLClient(
    credential,
    CONFIG["subscription_id"],
    CONFIG["resource_group"],
    CONFIG["workspace_name"],
)
workspace = ml_client.workspaces.get(CONFIG["workspace_name"])
registered_model = ml_client.models.get(CONFIG["model_name"], CONFIG["model_version"])
compute = ml_client.compute.get(CONFIG["compute_name"])
output_datastore = ml_client.datastores.get(CONFIG["output_datastore"])
if registered_model.type != AssetTypes.CUSTOM_MODEL:
    raise RuntimeError(f"Expected a custom_model, found {registered_model.type}")
if compute.provisioning_state != "Succeeded":
    raise RuntimeError(f"Compute state is {compute.provisioning_state}")
datastore_type = str(output_datastore.type).lower()
if "azure_data_lake_gen2" not in datastore_type:
    raise RuntimeError(f"Expected ADLS Gen2, found {output_datastore.type}")

print(f"Python:     {sys.version.split()[0]}")
print(f"Workspace:  {workspace.name}")
print(f"Model:      {registered_model.name}:{registered_model.version}")
print(f"Compute:    {compute.name} ({compute.provisioning_state})")
print(f"ADLS store: {output_datastore.name} ({output_datastore.type})")
{
    "experiment_name": CONFIG["experiment_name"],
    "schedule_name": CONFIG["schedule_name"],
    "submit_to_azure": CONFIG["submit_to_azure"],
    "create_test_schedule": CONFIG["create_test_schedule"],
    "disable_schedule_after_test": CONFIG["disable_schedule_after_test"],
    "delete_schedule_after_test": CONFIG["delete_schedule_after_test"],
}

## 2. Materialize the Batch Scoring Runtime

The notebook writes its complete batch runtime to a notebook-specific temporary directory. The command validates the registered binary bundle, supports both feature-ready and raw taxi CSV input, rejects invalid rows without logging their feature values, scores valid rows with one H2O JVM, and emits predictions plus aggregate monitoring artifacts.

MLflow receives counts, timing, throughput, prediction distribution, feature aggregates, lineage tags, and non-row-level artifacts. Raw input rows and prediction rows remain only in the explicit Azure ML outputs.

In [ ]:
SCORE_SOURCE = r'''
from __future__ import annotations

import argparse
import hashlib
import json
import math
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import h2o
import mlflow
import numpy as np
import pandas as pd


def parse_bool(value: str) -> bool:
    normalized = str(value).strip().lower()
    if normalized in {"1", "true", "yes", "y"}:
        return True
    if normalized in {"0", "false", "no", "n"}:
        return False
    raise argparse.ArgumentTypeError(f"Expected a boolean value, received {value!r}")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Score a CSV with an H2O binary model")
    parser.add_argument("--model-dir", required=True)
    parser.add_argument("--input-data", required=True)
    parser.add_argument("--scored-output", required=True)
    parser.add_argument("--monitoring-output", required=True)
    parser.add_argument("--correlation-id", required=True)
    parser.add_argument("--id-column", default="__generated__")
    parser.add_argument("--fail-on-rejects", type=parse_bool, default=False)
    return parser.parse_args()


def emit(event: str, **fields: object) -> None:
    print(json.dumps({"event": event, **fields}, sort_keys=True, default=str), flush=True)


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_one(root: Path, name: str) -> Path:
    matches = list(root.rglob(name))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {name} beneath {root}, found {len(matches)}")
    return matches[0]


def resolve_csv(path: Path) -> Path:
    if path.is_file():
        return path
    matches = sorted(path.rglob("*.csv"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one CSV beneath {path}, found {len(matches)}")
    return matches[0]


def safe_metric_name(value: str) -> str:
    return "".join(character if character.isalnum() else "_" for character in value).strip("_")


def prepare_input(
    input_path: Path,
    manifest: dict,
    id_column: str,
) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame, dict[str, int], str]:
    source_path = resolve_csv(input_path)
    source = pd.read_csv(source_path)
    if source.empty:
        raise ValueError("Input CSV contains no rows")

    if "pickupHour" not in source.columns and "tpepPickupDateTime" in source.columns:
        pickup_time = pd.to_datetime(source["tpepPickupDateTime"], errors="coerce")
        source["pickupHour"] = pickup_time.dt.hour

    features = manifest["features"]
    missing = [column for column in features if column not in source.columns]
    if missing:
        raise ValueError(f"Missing required feature columns: {missing}")

    if id_column and id_column != "__generated__":
        if id_column not in source.columns:
            raise ValueError(f"Configured ID column is missing: {id_column}")
        if source[id_column].isna().any():
            raise ValueError(f"Configured ID column contains null values: {id_column}")
        row_ids = source[id_column].astype(str)
        if row_ids.duplicated().any():
            raise ValueError(f"Configured ID column contains duplicates: {id_column}")
    else:
        width = max(6, len(str(len(source))))
        row_ids = pd.Series(
            [f"{source_path.stem}:{index:0{width}d}" for index in range(len(source))],
            index=source.index,
            dtype="string",
        )

    numeric = pd.DataFrame(index=source.index)
    invalid_by_feature: dict[str, int] = {}
    for feature in features:
        numeric[feature] = pd.to_numeric(source[feature], errors="coerce")
        invalid_by_feature[feature] = int((~np.isfinite(numeric[feature])).sum())

    invalid_mask = ~np.isfinite(numeric.to_numpy(dtype=float)).all(axis=1)
    rejection_reasons = []
    for row_index in numeric.index[invalid_mask]:
        invalid_features = [
            feature for feature in features if not math.isfinite(float(numeric.at[row_index, feature]))
        ]
        rejection_reasons.append("invalid_or_null:" + ",".join(invalid_features))

    rejects = pd.DataFrame(
        {
            "row_id": row_ids.loc[invalid_mask].astype(str).to_numpy(),
            "source_file": source_path.name,
            "status": "rejected",
            "rejection_reason": rejection_reasons,
        }
    )
    valid_mask = ~invalid_mask
    return (
        numeric.loc[valid_mask, features].reset_index(drop=True),
        row_ids.loc[valid_mask].reset_index(drop=True),
        rejects,
        invalid_by_feature,
        source_path.name,
    )


def log_mlflow(summary: dict, manifest: dict, feature_statistics: pd.DataFrame, output_dir: Path) -> str:
    if not os.getenv("AZUREML_RUN_ID"):
        return "skipped_local"
    try:
        mlflow.log_params(
            {
                "model_name": manifest["model_name"],
                "model_version": manifest["model_version"],
                "model_format": manifest["model_format"],
                "h2o_version": manifest["h2o_version"],
                "model_sha256": summary["model_sha256"],
                "correlation_id": summary["correlation_id"],
            }
        )
        metrics = {
            "input_rows": summary["input_rows"],
            "scored_rows": summary["scored_rows"],
            "rejected_rows": summary["rejected_rows"],
            "reject_rate": summary["reject_rate"],
            "duration_seconds": summary["duration_seconds"],
            "jvm_startup_seconds": summary["jvm_startup_seconds"],
            "scoring_seconds": summary["scoring_seconds"],
            "rows_per_second": summary["rows_per_second"],
        }
        if summary["scored_rows"]:
            metrics.update(
                {
                    "prediction_mean": summary["prediction_mean"],
                    "prediction_std": summary["prediction_std"],
                    "prediction_min": summary["prediction_min"],
                    "prediction_max": summary["prediction_max"],
                }
            )
        for _, row in feature_statistics.iterrows():
            prefix = "feature_" + safe_metric_name(str(row["feature"]))
            for statistic in ("mean", "std", "min", "max", "invalid_count"):
                value = row.get(statistic)
                if pd.notna(value):
                    metrics[f"{prefix}_{statistic}"] = float(value)
        mlflow.log_metrics(metrics)
        mlflow.set_tags(
            {
                "pipeline_stage": "h2o_batch_scoring",
                "model_format": "h2o_binary",
                "aml_run_id": summary["aml_run_id"],
                "source_file": summary["source_file"],
                "quality_gate": summary["quality_gate"],
            }
        )
        mlflow.log_artifact(str(output_dir / "feature_statistics.csv"), "monitoring")
        mlflow.log_artifact(str(output_dir / "run_manifest.json"), "monitoring")
        return "logged"
    except Exception as exc:
        emit("mlflow_logging_failed", error_type=type(exc).__name__)
        return "failed"


def main() -> None:
    args = parse_args()
    started = time.perf_counter()
    scoring_time = datetime.now(timezone.utc).isoformat()
    aml_run_id = os.getenv("AZUREML_RUN_ID", "local")
    correlation_id = args.correlation_id or aml_run_id
    scored_output = Path(args.scored_output)
    monitoring_output = Path(args.monitoring_output)
    scored_output.mkdir(parents=True, exist_ok=True)
    monitoring_output.mkdir(parents=True, exist_ok=True)

    model_root = Path(args.model_dir).resolve()
    manifest_path = find_one(model_root, "model_manifest.json")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    model_path = manifest_path.parent / manifest["model_file"]
    if manifest.get("model_format") != "h2o_binary":
        raise RuntimeError("Registered model is not an H2O binary model")
    if h2o.__version__ != manifest["h2o_version"]:
        raise RuntimeError(f"Expected h2o=={manifest['h2o_version']}, found {h2o.__version__}")
    model_sha = sha256(model_path)
    if model_sha != manifest["files"].get(model_path.name):
        raise RuntimeError("Binary model checksum does not match the manifest")

    valid_features, valid_ids, rejects, invalid_counts, source_file = prepare_input(
        Path(args.input_data), manifest, args.id_column
    )
    input_rows = len(valid_features) + len(rejects)
    emit(
        "batch_scoring_started",
        aml_run_id=aml_run_id,
        correlation_id=correlation_id,
        input_rows=input_rows,
        valid_rows=len(valid_features),
        rejected_rows=len(rejects),
        model_name=manifest["model_name"],
        model_version=manifest["model_version"],
    )

    predictions = np.asarray([], dtype=float)
    jvm_startup_seconds = 0.0
    scoring_seconds = 0.0
    try:
        if len(valid_features):
            h2o.no_progress()
            jvm_started = time.perf_counter()
            h2o.init(
                ip="127.0.0.1",
                port=54321,
                start_h2o=True,
                nthreads=int(os.getenv("H2O_NTHREADS", "4")),
                max_mem_size=os.getenv("H2O_MAX_MEM_SIZE", "6G"),
                strict_version_check=True,
                bind_to_localhost=True,
                verbose=False,
                telemetry=False,
            )
            model = h2o.load_model(str(model_path))
            jvm_startup_seconds = time.perf_counter() - jvm_started
            scoring_started = time.perf_counter()
            h2o_frame = h2o.H2OFrame(valid_features)
            for column in manifest.get("categorical_features", []):
                h2o_frame[column] = h2o_frame[column].asfactor()
            prediction_frame = model.predict(h2o_frame)
            predictions = prediction_frame.as_data_frame()["predict"].to_numpy(dtype=float)
            scoring_seconds = time.perf_counter() - scoring_started
            h2o.remove(prediction_frame)
            h2o.remove(h2o_frame)
    finally:
        try:
            if h2o.connection() is not None:
                h2o.cluster().shutdown(prompt=False)
        except Exception as exc:
            emit("h2o_shutdown_failed", error_type=type(exc).__name__)

    predictions_frame = pd.DataFrame(
        {
            "row_id": valid_ids.astype(str),
            "prediction": predictions,
            "model_name": manifest["model_name"],
            "model_version": manifest["model_version"],
            "model_sha256": model_sha,
            "h2o_version": manifest["h2o_version"],
            "aml_run_id": aml_run_id,
            "source_file": source_file,
            "scoring_time_utc": scoring_time,
            "correlation_id": correlation_id,
            "status": "scored",
        }
    )
    if len(predictions_frame) != len(valid_features):
        raise RuntimeError("Prediction count does not match valid input row count")

    monitoring_frame = valid_features.copy()
    monitoring_frame.insert(0, "row_id", valid_ids.astype(str))
    monitoring_frame["prediction"] = predictions
    monitoring_frame["model_name"] = manifest["model_name"]
    monitoring_frame["model_version"] = manifest["model_version"]
    monitoring_frame["scoring_time_utc"] = scoring_time
    monitoring_frame["correlation_id"] = correlation_id

    statistics = valid_features.describe().T.reset_index(names="feature") if len(valid_features) else pd.DataFrame({"feature": manifest["features"]})
    statistics["invalid_count"] = statistics["feature"].map(invalid_counts).fillna(0).astype(int)

    predictions_frame.to_csv(scored_output / "predictions.csv", index=False)
    rejects.to_csv(scored_output / "rejects.csv", index=False)
    monitoring_frame.to_csv(monitoring_output / "monitoring_data.csv", index=False)
    statistics.to_csv(monitoring_output / "feature_statistics.csv", index=False)

    duration_seconds = time.perf_counter() - started
    summary = {
        "aml_run_id": aml_run_id,
        "correlation_id": correlation_id,
        "source_file": source_file,
        "model_name": manifest["model_name"],
        "model_version": manifest["model_version"],
        "model_sha256": model_sha,
        "h2o_version": manifest["h2o_version"],
        "input_rows": input_rows,
        "scored_rows": len(predictions_frame),
        "rejected_rows": len(rejects),
        "reject_rate": len(rejects) / input_rows,
        "duration_seconds": duration_seconds,
        "jvm_startup_seconds": jvm_startup_seconds,
        "scoring_seconds": scoring_seconds,
        "rows_per_second": len(predictions_frame) / duration_seconds if duration_seconds else 0.0,
        "prediction_mean": float(predictions.mean()) if len(predictions) else None,
        "prediction_std": float(predictions.std()) if len(predictions) else None,
        "prediction_min": float(predictions.min()) if len(predictions) else None,
        "prediction_max": float(predictions.max()) if len(predictions) else None,
        "quality_gate": "passed" if len(predictions_frame) + len(rejects) == input_rows else "failed",
        "mlflow_status": "pending",
    }
    run_manifest = {
        "schema_version": "1.0",
        "created_utc": scoring_time,
        "input": {"source_file": source_file, "rows": input_rows},
        "model": {
            "name": manifest["model_name"],
            "version": manifest["model_version"],
            "format": manifest["model_format"],
            "sha256": model_sha,
            "h2o_version": manifest["h2o_version"],
        },
        "execution": {
            "aml_run_id": aml_run_id,
            "correlation_id": correlation_id,
            "fail_on_rejects": args.fail_on_rejects,
        },
        "outputs": {
            "predictions": "predictions.csv",
            "rejects": "rejects.csv",
            "monitoring_data": "monitoring_data.csv",
            "feature_statistics": "feature_statistics.csv",
        },
    }
    (monitoring_output / "run_manifest.json").write_text(
        json.dumps(run_manifest, indent=2) + "\n", encoding="utf-8"
    )
    summary["mlflow_status"] = log_mlflow(summary, manifest, statistics, monitoring_output)
    (monitoring_output / "summary.json").write_text(
        json.dumps(summary, indent=2) + "\n", encoding="utf-8"
    )
    if os.getenv("AZUREML_RUN_ID") and summary["mlflow_status"] == "logged":
        mlflow.log_artifact(str(monitoring_output / "summary.json"), "monitoring")

    emit(
        "batch_scoring_completed",
        aml_run_id=aml_run_id,
        correlation_id=correlation_id,
        input_rows=input_rows,
        scored_rows=len(predictions_frame),
        rejected_rows=len(rejects),
        duration_seconds=round(duration_seconds, 6),
        rows_per_second=round(summary["rows_per_second"], 6),
        quality_gate=summary["quality_gate"],
        mlflow_status=summary["mlflow_status"],
    )
    if args.fail_on_rejects and len(rejects):
        raise RuntimeError(f"Rejected {len(rejects)} rows and fail_on_rejects is enabled")


if __name__ == "__main__":
    main()
'''.lstrip()

CONDA_SOURCE = '''
name: h2o-binary-batch
channels:
  - conda-forge
dependencies:
  - python=3.12
  - openjdk=17
  - pip
  - pip:
      - h2o==3.46.0.12
      - numpy==1.26.4
      - pandas==2.2.3
      - mlflow==2.22.1
      - azureml-mlflow==1.60.0.post1
'''.lstrip()

SCORE_PATH.write_text(SCORE_SOURCE, encoding="utf-8")
CONDA_PATH.write_text(CONDA_SOURCE, encoding="utf-8")
compile(SCORE_SOURCE, str(SCORE_PATH), "exec")
print(f"Scorer:      {SCORE_PATH}")
print(f"Environment: {CONDA_PATH}")
print(f"Python lines: {len(SCORE_SOURCE.splitlines())}")

## 3. Prove the Batch Contract Locally

This is the cheapest discriminating test for the pipeline. The generated command runs against the Notebook 01 model bundle and golden CSV, then its file outputs are compared to the original H2O predictions. Azure submission remains blocked until this passes.

In [ ]:
if LOCAL_OUTPUT_DIR.exists():
    shutil.rmtree(LOCAL_OUTPUT_DIR)
local_scored_dir = LOCAL_OUTPUT_DIR / "scored"
local_monitoring_dir = LOCAL_OUTPUT_DIR / "monitoring"
local_scored_dir.mkdir(parents=True)
local_monitoring_dir.mkdir(parents=True)

local_environment = os.environ.copy()
if not shutil.which("java", path=local_environment.get("PATH")):
    java_bins = sorted((Path.home() / ".jdk").glob("jdk-17*/bin"), reverse=True)
    if not java_bins:
        raise FileNotFoundError("OpenJDK 17 was not found on PATH or beneath ~/.jdk")
    local_environment["JAVA_HOME"] = str(java_bins[0].parent)
    local_environment["PATH"] = str(java_bins[0]) + os.pathsep + local_environment["PATH"]

local_command = [
    sys.executable,
    str(SCORE_PATH),
    "--model-dir",
    str(MODEL_DIR),
    "--input-data",
    str(MODEL_DIR / "golden_input.csv"),
    "--scored-output",
    str(local_scored_dir),
    "--monitoring-output",
    str(local_monitoring_dir),
    "--correlation-id",
    "notebook-05-local-golden",
    "--id-column",
    "",
    "--fail-on-rejects",
    "false",
]
local_result = subprocess.run(
    local_command,
    env=local_environment,
    capture_output=True,
    text=True,
    timeout=180,
    check=False,
)
print(local_result.stdout)
if local_result.returncode:
    print(local_result.stderr)
    raise RuntimeError(f"Local batch scorer failed with exit code {local_result.returncode}")

local_predictions = pd.read_csv(local_scored_dir / "predictions.csv")
local_rejects = pd.read_csv(local_scored_dir / "rejects.csv")
local_summary = json.loads((local_monitoring_dir / "summary.json").read_text(encoding="utf-8"))
golden_expected = pd.read_csv(MODEL_DIR / "golden_expected.csv")
np.testing.assert_allclose(
    golden_expected["predict"].to_numpy(dtype=float),
    local_predictions["prediction"].to_numpy(dtype=float),
    rtol=1e-6,
    atol=1e-6,
)
assert local_summary["input_rows"] == len(golden_expected)
assert local_summary["scored_rows"] == len(golden_expected)
assert local_summary["rejected_rows"] == 0
assert local_summary["quality_gate"] == "passed"
assert local_summary["mlflow_status"] == "skipped_local"
assert local_rejects.empty
required_outputs = {
    local_scored_dir / "predictions.csv",
    local_scored_dir / "rejects.csv",
    local_monitoring_dir / "monitoring_data.csv",
    local_monitoring_dir / "feature_statistics.csv",
    local_monitoring_dir / "summary.json",
    local_monitoring_dir / "run_manifest.json",
}
assert all(path.is_file() for path in required_outputs)

comparison = pd.DataFrame(
    {
        "expected": golden_expected["predict"],
        "batch_scorer": local_predictions["prediction"],
        "absolute_error": np.abs(
            golden_expected["predict"].to_numpy(dtype=float)
            - local_predictions["prediction"].to_numpy(dtype=float)
        ),
    }
)
print(f"Local rows reconciled: {local_summary['input_rows']}")
print(f"Maximum absolute error: {comparison['absolute_error'].max():.10f}")
display(local_summary)
display(comparison.head(10))

## 4. Register the Pinned Environment and Demonstration Input

The H2O binary format is version-specific, so the batch environment is immutable and records the exact H2O runtime. The checked-in raw taxi CSV is registered as a versioned `uri_file`; scheduled runs therefore reference a durable Azure asset rather than this workstation.

In [ ]:
environment_definition = Environment(
    name=CONFIG["environment_name"],
    version=CONFIG["environment_version"],
    description="OpenJDK 17 and H2O 3.46.0.12 for monitored H2O binary-model batch scoring",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
    conda_file=str(CONDA_PATH),
    tags={
        "model_format": "h2o_binary",
        "h2o_version": "3.46.0.12",
        "python_version": "3.12",
        "telemetry": "mlflow_aggregate_only",
    },
)
data_definition = Data(
    name=CONFIG["data_name"],
    version=CONFIG["data_version"],
    description="Checked-in yellow taxi CSV for the H2O batch-scoring workshop",
    type=AssetTypes.URI_FILE,
    path=str(DATA_PATH),
    tags={"purpose": "batch_scoring_demo", "contains_labels": "true"},
)

if CONFIG["submit_to_azure"]:
    try:
        registered_environment = ml_client.environments.get(
            CONFIG["environment_name"], CONFIG["environment_version"]
        )
        if registered_environment.tags.get("h2o_version") != "3.46.0.12":
            raise RuntimeError("Existing environment version has an unexpected H2O tag")
        print(f"Using existing environment: {registered_environment.id}")
    except ResourceNotFoundError:
        registered_environment = ml_client.environments.create_or_update(environment_definition)
        print(f"Registered environment: {registered_environment.id}")

    try:
        registered_data = ml_client.data.get(CONFIG["data_name"], CONFIG["data_version"])
        print(f"Using existing input data: {registered_data.id}")
    except ResourceNotFoundError:
        registered_data = ml_client.data.create_or_update(data_definition)
        print(f"Registered input data: {registered_data.id}")
else:
    registered_environment = environment_definition
    registered_data = data_definition
    print("Azure mutation disabled; environment and data definitions were created locally.")

assert registered_environment.name == CONFIG["environment_name"]
assert str(registered_environment.version) == CONFIG["environment_version"]
assert registered_data.name == CONFIG["data_name"]
assert str(registered_data.version) == CONFIG["data_version"]

## 5. Define the Reusable Scoring Pipeline Inline

The pipeline exposes the input file, model asset, correlation ID, optional business ID column, and reject policy. Both output folders are first-class pipeline outputs. Azure ML assigns each run a unique path in `datacollection_adls`, so manual and scheduled runs do not overwrite one another.

In [ ]:
score_component = command(
    name="h2o_binary_batch_score_05",
    display_name="Score H2O binary model",
    description="Validate, score, reconcile, and emit aggregate telemetry for an H2O binary model",
    code=str(CODE_DIR),
    command=(
        "python score_batch.py "
        "--model-dir ${{inputs.model_dir}} "
        "--input-data ${{inputs.input_data}} "
        "--scored-output ${{outputs.scored_output}} "
        "--monitoring-output ${{outputs.monitoring_output}} "
        "--correlation-id '${{inputs.correlation_id}}' "
        "--id-column '${{inputs.id_column}}' "
        "--fail-on-rejects ${{inputs.fail_on_rejects}}"
    ),
    environment=f"azureml:{CONFIG['environment_name']}:{CONFIG['environment_version']}",
    identity=ManagedIdentityConfiguration(client_id=CONFIG["compute_identity_client_id"]),
    environment_variables={
        "H2O_NTHREADS": str(CONFIG["h2o_threads"]),
        "H2O_MAX_MEM_SIZE": CONFIG["h2o_heap"],
    },
    inputs={
        "model_dir": Input(type=AssetTypes.CUSTOM_MODEL),
        "input_data": Input(type=AssetTypes.URI_FILE),
        "correlation_id": Input(type="string"),
        "id_column": Input(type="string"),
        "fail_on_rejects": Input(type="boolean"),
    },
    outputs={
        "scored_output": Output(type=AssetTypes.URI_FOLDER, mode="rw_mount"),
        "monitoring_output": Output(type=AssetTypes.URI_FOLDER, mode="rw_mount"),
    },
    is_deterministic=False,
)


@pipeline(
    name="h2o_binary_batch_scoring_05",
    description="H2O binary-model batch scoring with ADLS outputs and aggregate monitoring",
)
def build_h2o_scoring_pipeline(
    model_dir,
    input_data,
    correlation_id,
    id_column,
    fail_on_rejects,
):
    scoring_step = score_component(
        model_dir=model_dir,
        input_data=input_data,
        correlation_id=correlation_id,
        id_column=id_column,
        fail_on_rejects=fail_on_rejects,
    )
    return {
        "scored_output": scoring_step.outputs.scored_output,
        "monitoring_output": scoring_step.outputs.monitoring_output,
    }


manual_correlation_id = f"manual-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}-{uuid.uuid4().hex[:8]}"
pipeline_job = build_h2o_scoring_pipeline(
    model_dir=Input(
        type=AssetTypes.CUSTOM_MODEL,
        path=f"azureml:{CONFIG['model_name']}:{CONFIG['model_version']}",
    ),
    input_data=Input(
        type=AssetTypes.URI_FILE,
        path=f"azureml:{CONFIG['data_name']}:{CONFIG['data_version']}",
    ),
    correlation_id=manual_correlation_id,
    id_column="__generated__",
    fail_on_rejects=False,
)
pipeline_job.display_name = "H2O taxi batch scoring - manual validation"
pipeline_job.experiment_name = CONFIG["experiment_name"]
pipeline_job.tags = {
    "model": f"{CONFIG['model_name']}:{CONFIG['model_version']}",
    "model_format": "h2o_binary",
    "trigger": "manual",
    "correlation_id": manual_correlation_id,
    "output_datastore": CONFIG["output_datastore"],
}
pipeline_job.settings.default_compute = CONFIG["compute_name"]
pipeline_job.settings.default_datastore = CONFIG["output_datastore"]
pipeline_job.settings.continue_on_step_failure = False
pipeline_job.settings.force_rerun = True

assert pipeline_job.inputs["correlation_id"]._data == manual_correlation_id
assert pipeline_job.inputs["id_column"]._data == "__generated__"
assert pipeline_job.settings.default_compute == CONFIG["compute_name"]
assert pipeline_job.settings.default_datastore == CONFIG["output_datastore"]
print(f"Correlation ID:   {manual_correlation_id}")
print(f"Default compute:  {pipeline_job.settings.default_compute}")
print(f"Default datastore:{pipeline_job.settings.default_datastore}")
print(f"Pipeline outputs: {list(pipeline_job.outputs)}")

## 6. Submit and Stream the Manual Pipeline Run

This run validates the complete cloud path before scheduling: immutable assets, managed-identity data access, environment construction, compute startup, H2O scoring, MLflow logging, and ADLS output publication.

In [ ]:
if CONFIG["submit_to_azure"]:
    submitted_job = ml_client.jobs.create_or_update(
        pipeline_job,
        experiment_name=CONFIG["experiment_name"],
    )
    print(f"Submitted pipeline: {submitted_job.name}")
    if submitted_job.services and "Studio" in submitted_job.services:
        print(f"Studio: {submitted_job.services['Studio'].endpoint}")
    ml_client.jobs.stream(submitted_job.name)
    submitted_job = ml_client.jobs.get(submitted_job.name)
    if submitted_job.status != "Completed":
        raise RuntimeError(f"Manual pipeline ended with status {submitted_job.status}")
    child_jobs = list(ml_client.jobs.list(parent_job_name=submitted_job.name))
    if len(child_jobs) != 1:
        raise RuntimeError(f"Expected one scoring child job, found {len(child_jobs)}")
    scoring_job = child_jobs[0]
    if scoring_job.status != "Completed":
        raise RuntimeError(f"Scoring child ended with status {scoring_job.status}")
    print(f"Manual pipeline status: {submitted_job.status}")
    print(f"Scoring child: {scoring_job.name} ({scoring_job.status})")
    for output_name, output in submitted_job.outputs.items():
        print(f"{output_name}: {output.path}")
else:
    submitted_job = None
    scoring_job = None
    print("Azure submission disabled; local golden validation remains authoritative.")

## 7. Validate Outputs, Lineage, and MLflow Monitoring

The private ADLS account intentionally rejects direct access from this workstation. Validation therefore uses the Azure ML job state and MLflow tracking metadata, while the row-level output URIs remain private. This preserves the network boundary and avoids copying predictions into notebook output or telemetry.

In [ ]:
if CONFIG["submit_to_azure"]:
    mlflow.set_tracking_uri(workspace.mlflow_tracking_uri)
    mlflow_client = MlflowClient()
    tracked_run = mlflow_client.get_run(scoring_job.name)
    metrics = tracked_run.data.metrics
    parameters = tracked_run.data.params
    monitoring_artifacts = {
        artifact.path for artifact in mlflow_client.list_artifacts(scoring_job.name, "monitoring")
    }
    expected_artifacts = {
        "monitoring/feature_statistics.csv",
        "monitoring/run_manifest.json",
        "monitoring/summary.json",
    }
    source_row_count = len(pd.read_csv(DATA_PATH))
    assert tracked_run.info.status == "FINISHED"
    assert int(metrics["input_rows"]) == source_row_count
    assert int(metrics["scored_rows"]) + int(metrics["rejected_rows"]) == source_row_count
    assert tracked_run.data.tags["quality_gate"] == "passed"
    assert parameters["model_name"] == CONFIG["model_name"]
    assert parameters["model_version"] == CONFIG["model_version"]
    assert expected_artifacts.issubset(monitoring_artifacts)

    output_base_uri = (
        f"azureml://subscriptions/{CONFIG['subscription_id']}"
        f"/resourcegroups/{CONFIG['resource_group']}"
        f"/workspaces/{CONFIG['workspace_name']}"
        f"/datastores/{CONFIG['output_datastore']}"
        f"/paths/azureml/{scoring_job.name}"
    )
    manual_scored_uri = f"{output_base_uri}/scored_output/"
    manual_monitoring_uri = f"{output_base_uri}/monitoring_output/"
    manual_studio_url = (
        submitted_job.services["Studio"].endpoint
        if submitted_job.services and "Studio" in submitted_job.services
        else submitted_job.studio_url
    )
    monitoring_table = pd.DataFrame(
        [
            {"metric": "input_rows", "value": int(metrics["input_rows"])},
            {"metric": "scored_rows", "value": int(metrics["scored_rows"])},
            {"metric": "rejected_rows", "value": int(metrics["rejected_rows"])},
            {"metric": "reject_rate", "value": metrics["reject_rate"]},
            {"metric": "duration_seconds", "value": metrics["duration_seconds"]},
            {"metric": "rows_per_second", "value": metrics["rows_per_second"]},
            {"metric": "prediction_mean", "value": metrics["prediction_mean"]},
            {"metric": "prediction_std", "value": metrics["prediction_std"]},
        ]
    )
    print(f"Scored output:     {manual_scored_uri}")
    print(f"Monitoring output: {manual_monitoring_uri}")
    print(f"MLflow artifacts:  {sorted(monitoring_artifacts)}")
    display(monitoring_table)
else:
    metrics = local_summary
    manual_scored_uri = str(local_scored_dir)
    manual_monitoring_uri = str(local_monitoring_dir)
    manual_studio_url = None
    print("Using local validation outputs because Azure submission is disabled.")

## 8. Create and Prove a Native Azure ML Schedule

The test uses a one-minute UTC recurrence so the workshop can prove scheduling without waiting for a business-time cron window. The notebook detects the first scheduled job, immediately disables the schedule, validates the completed child run and metrics, and leaves the disabled resource for workshop inspection unless deletion is explicitly enabled.

For production, replace this test recurrence with the approved cron or business recurrence while keeping the same pipeline job definition.

In [ ]:
from azure.ai.ml.entities import RecurrenceTrigger

scheduled_job = None
scheduled_child = None
scheduled_metrics = None
schedule_scored_uri = None
schedule_monitoring_uri = None
created_schedule = None

if CONFIG["submit_to_azure"] and CONFIG["create_test_schedule"]:
    scheduled_correlation_id = (
        f"{CONFIG['schedule_name']}-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}"
    )
    scheduled_pipeline_job = build_h2o_scoring_pipeline(
        model_dir=Input(
            type=AssetTypes.CUSTOM_MODEL,
            path=f"azureml:{CONFIG['model_name']}:{CONFIG['model_version']}",
        ),
        input_data=Input(
            type=AssetTypes.URI_FILE,
            path=f"azureml:{CONFIG['data_name']}:{CONFIG['data_version']}",
        ),
        correlation_id=scheduled_correlation_id,
        id_column="__generated__",
        fail_on_rejects=False,
    )
    scheduled_pipeline_job.display_name = "H2O taxi batch scoring - native schedule"
    scheduled_pipeline_job.experiment_name = CONFIG["experiment_name"]
    scheduled_pipeline_job.tags = {
        "model": f"{CONFIG['model_name']}:{CONFIG['model_version']}",
        "model_format": "h2o_binary",
        "trigger": "native_schedule",
        "schedule_name": CONFIG["schedule_name"],
        "correlation_id": scheduled_correlation_id,
        "output_datastore": CONFIG["output_datastore"],
    }
    scheduled_pipeline_job.settings.default_compute = CONFIG["compute_name"]
    scheduled_pipeline_job.settings.default_datastore = CONFIG["output_datastore"]
    scheduled_pipeline_job.settings.continue_on_step_failure = False
    scheduled_pipeline_job.settings.force_rerun = True

    known_job_names = {job.name for job in ml_client.jobs.list()}
    try:
        existing_schedule = ml_client.schedules.get(CONFIG["schedule_name"])
        ml_client.schedules.begin_disable(name=existing_schedule.name).result()
        print(f"Disabled existing schedule before replacement: {existing_schedule.name}")
    except ResourceNotFoundError:
        pass

    trigger = RecurrenceTrigger(frequency="minute", interval=1, time_zone=TimeZone.UTC)
    schedule_definition = JobSchedule(
        name=CONFIG["schedule_name"],
        display_name="H2O taxi batch scoring test schedule",
        description="Short-lived workshop validation of native Azure ML pipeline scheduling",
        trigger=trigger,
        create_job=scheduled_pipeline_job,
        tags={
            "purpose": "workshop_validation",
            "model": f"{CONFIG['model_name']}:{CONFIG['model_version']}",
            "disable_after_test": "true",
        },
    )

    try:
        created_schedule = ml_client.schedules.begin_create_or_update(
            schedule=schedule_definition
        ).result()
        created_schedule = ml_client.schedules.begin_enable(
            name=CONFIG["schedule_name"]
        ).result()
        print(f"Schedule enabled: {created_schedule.name}")
        print("Recurrence: every 1 minute (UTC)")

        deadline = time.monotonic() + CONFIG["schedule_wait_minutes"] * 60
        while time.monotonic() < deadline and scheduled_job is None:
            for candidate in ml_client.jobs.list():
                candidate_tags = candidate.tags or {}
                is_new = candidate.name not in known_job_names
                belongs_to_schedule = (
                    candidate_tags.get("schedule_name") == CONFIG["schedule_name"]
                    or candidate_tags.get("correlation_id") == scheduled_correlation_id
                    or candidate.display_name == scheduled_pipeline_job.display_name
                )
                if is_new and belongs_to_schedule:
                    scheduled_job = candidate
                    break
            if scheduled_job is None:
                time.sleep(15)

        if scheduled_job is None:
            raise TimeoutError(
                f"No scheduled job appeared within {CONFIG['schedule_wait_minutes']} minutes"
            )
        print(f"Scheduled pipeline detected: {scheduled_job.name}")

        if CONFIG["disable_schedule_after_test"]:
            created_schedule = ml_client.schedules.begin_disable(
                name=CONFIG["schedule_name"]
            ).result()
            print(f"Schedule disabled: {created_schedule.name}")

        ml_client.jobs.stream(scheduled_job.name)
        scheduled_job = ml_client.jobs.get(scheduled_job.name)
        if scheduled_job.status != "Completed":
            raise RuntimeError(f"Scheduled pipeline ended with status {scheduled_job.status}")
        scheduled_children = list(ml_client.jobs.list(parent_job_name=scheduled_job.name))
        if len(scheduled_children) != 1:
            raise RuntimeError(f"Expected one scheduled scoring child, found {len(scheduled_children)}")
        scheduled_child = scheduled_children[0]
        if scheduled_child.status != "Completed":
            raise RuntimeError(f"Scheduled child ended with status {scheduled_child.status}")

        scheduled_run = mlflow_client.get_run(scheduled_child.name)
        scheduled_metrics = scheduled_run.data.metrics
        assert scheduled_run.info.status == "FINISHED"
        assert int(scheduled_metrics["input_rows"]) == source_row_count
        assert int(scheduled_metrics["scored_rows"]) + int(
            scheduled_metrics["rejected_rows"]
        ) == source_row_count
        assert scheduled_run.data.tags["quality_gate"] == "passed"

        schedule_output_base = (
            f"azureml://subscriptions/{CONFIG['subscription_id']}"
            f"/resourcegroups/{CONFIG['resource_group']}"
            f"/workspaces/{CONFIG['workspace_name']}"
            f"/datastores/{CONFIG['output_datastore']}"
            f"/paths/azureml/{scheduled_child.name}"
        )
        schedule_scored_uri = f"{schedule_output_base}/scored_output/"
        schedule_monitoring_uri = f"{schedule_output_base}/monitoring_output/"
        print(f"Scheduled status: {scheduled_job.status}")
        print(f"Scheduled scored output: {schedule_scored_uri}")
        print(f"Scheduled monitoring output: {schedule_monitoring_uri}")
    finally:
        if created_schedule is not None and CONFIG["disable_schedule_after_test"]:
            try:
                created_schedule = ml_client.schedules.begin_disable(
                    name=CONFIG["schedule_name"]
                ).result()
            except Exception as disable_error:
                print(f"Schedule disable retry failed: {disable_error}")
        if created_schedule is not None and CONFIG["delete_schedule_after_test"]:
            ml_client.schedules.begin_delete(name=CONFIG["schedule_name"]).result()
            print(f"Deleted schedule: {CONFIG['schedule_name']}")
else:
    print("Native schedule creation disabled.")

## 9. Monitoring and Operations Summary

Azure ML pipeline and batch workloads are monitored as jobs, not as synchronous endpoint requests. The strongest built-in evidence is the pipeline and child status, Azure ML Studio lineage, MLflow metrics and artifacts, command logs, compute utilization, and immutable ADLS outputs. Workspace Azure Monitor metrics, Activity Logs, diagnostic settings, and alerts complement those records.

Application Insights does not automatically create online-endpoint-style request telemetry for each scheduled or batch invocation. This implementation deliberately logs only aggregate lifecycle data to MLflow and stdout. Row-level features and predictions remain in the protected ADLS output folders.

In [ ]:
final_schedule = (
    ml_client.schedules.get(CONFIG["schedule_name"])
    if CONFIG["submit_to_azure"] and CONFIG["create_test_schedule"]
    else None
)
if final_schedule is not None and CONFIG["disable_schedule_after_test"]:
    assert final_schedule.is_enabled is False

summary = {
    "local_golden_parity": True,
    "local_golden_rows": local_summary["scored_rows"],
    "manual_pipeline_job": submitted_job.name if submitted_job else None,
    "manual_pipeline_status": submitted_job.status if submitted_job else "not_submitted",
    "manual_child_job": scoring_job.name if scoring_job else None,
    "manual_input_rows": int(metrics["input_rows"]),
    "manual_scored_rows": int(metrics["scored_rows"]),
    "manual_rejected_rows": int(metrics["rejected_rows"]),
    "manual_rows_per_second": metrics["rows_per_second"],
    "manual_studio_url": manual_studio_url,
    "manual_scored_output": manual_scored_uri,
    "manual_monitoring_output": manual_monitoring_uri,
    "scheduled_pipeline_job": scheduled_job.name if scheduled_job else None,
    "scheduled_pipeline_status": scheduled_job.status if scheduled_job else "not_created",
    "scheduled_child_job": scheduled_child.name if scheduled_child else None,
    "scheduled_scored_rows": int(scheduled_metrics["scored_rows"]) if scheduled_metrics else None,
    "scheduled_rejected_rows": int(scheduled_metrics["rejected_rows"]) if scheduled_metrics else None,
    "scheduled_rows_per_second": scheduled_metrics["rows_per_second"] if scheduled_metrics else None,
    "scheduled_scored_output": schedule_scored_uri,
    "scheduled_monitoring_output": schedule_monitoring_uri,
    "schedule_name": final_schedule.name if final_schedule else None,
    "schedule_enabled": final_schedule.is_enabled if final_schedule else None,
    "model": f"{CONFIG['model_name']}:{CONFIG['model_version']}",
    "environment": f"{CONFIG['environment_name']}:{CONFIG['environment_version']}",
    "input_data": f"{CONFIG['data_name']}:{CONFIG['data_version']}",
    "compute_identity_client_id": CONFIG["compute_identity_client_id"],
    "output_datastore": CONFIG["output_datastore"],
    "monitoring_model": "AML jobs + MLflow aggregates + logs + Azure Monitor",
    "row_level_telemetry": "ADLS outputs only",
}
display(summary)

if CONFIG["delete_schedule_after_test"] and final_schedule is not None:
    if final_schedule.is_enabled:
        ml_client.schedules.begin_disable(name=final_schedule.name).result()
    ml_client.schedules.begin_delete(name=final_schedule.name).result()
    print(f"Deleted schedule: {final_schedule.name}")
else:
    print("The validated schedule remains disabled for workshop inspection.")
print("Notebook 05 is complete. Notebook 06 can recreate this pipeline independently behind a batch endpoint.")